# C-MAPSS FD001 — RUL Benchmark on Kaggle 2×T4

Full 20 631-row training set, 8 chains × 300 sweeps, auto-pmap across both T4 GPUs.

All the heavy lifting lives in [`examples/c_mapss/`](examples/c_mapss/) scripts on the main repo —
this notebook just wires them up. `run_inference.py` auto-detects `jax.device_count() > 1`
and switches to `jax.pmap` with `chains // n_devices` chains per GPU.

**Kaggle setup:**
1. Settings → Accelerator → GPU T4 ×2
2. Settings → Internet → on
3. Run all

In [ ]:
# 1. Clone repo + install no-deps (preserves Kaggle's JAX+CUDA stack)
import os

WORKDIR = "/kaggle/working/jaxcross"
BRANCH = "feat/cmapss-pdm-example"  # change to 'main' after merge

!git clone https://github.com/sambhal-labs/jaxcross.git {WORKDIR} 2>/dev/null \
    || (cd {WORKDIR} && git fetch && git reset --hard)
os.chdir(WORKDIR)
!git fetch origin && (git checkout {BRANCH} || git checkout -b {BRANCH} origin/{BRANCH}) && git pull origin {BRANCH}

%pip install -e . --no-deps -q
%pip install polars scikit-learn -q  # polars + sklearn (belt-and-suspenders)

print(f"Branch: {BRANCH}")
print(f"CWD: {os.getcwd()}")

In [ ]:
# 2. Verify GPUs
import jax

print("JAX:", jax.__version__)
print("backend:", jax.default_backend())
print("devices:", jax.devices())
print("device_count:", jax.device_count())
assert jax.device_count() == 2, "Expected 2 T4 GPUs — check accelerator settings"

In [ ]:
# 3. Fetch + preprocess FD001 (whole dataset)
!python examples/c_mapss/fetch_cmapss.py
!python examples/c_mapss/preprocess_cmapss.py FD001

In [ ]:
# 4. Inference — 8 chains (4 per T4) x 300 sweeps on ALL 20 631 rows
#    No --subsample flag, so the full training set is used.
#    run_inference.py auto-detects 2 devices and switches to jax.pmap.
!python examples/c_mapss/run_inference.py FD001 \
    --chains 8 --sweeps 300 --diag-every 20 \
    --max-views 16 --max-clusters 32 --seed 42

In [ ]:
# 5. Evaluate RUL with BMA over all 8 chains + 90/95/99% CI coverage
!python examples/c_mapss/evaluate_rul.py FD001 --samples 1000

In [ ]:
# 6. Same-data sklearn baselines (Ridge + RandomForest) for apples-to-apples reference
!python examples/c_mapss/baseline_rul.py FD001

In [ ]:
# 7. Also evaluate best-chain-only (highest log_joint) for the BMA-vs-best comparison
!python examples/c_mapss/evaluate_best_chain.py FD001 --samples 1000

In [ ]:
# 8. Consolidated comparison table vs published baselines
import json
from pathlib import Path

EVAL = Path("examples/c_mapss/results/evaluation/FD001")
BASE = Path("examples/c_mapss/results/baselines/FD001")
INF = Path("examples/c_mapss/results/inference/FD001")

bma = json.loads((EVAL / "metrics.json").read_text())
best = json.loads((EVAL / "best_chain_metrics.json").read_text())
baselines = json.loads((BASE / "baseline_metrics.json").read_text())
meta = json.loads((INF / "inference_meta.json").read_text())

print(
    f"Inference: {meta['n_chains']} chains x {meta['n_sweeps']} sweeps, "
    f"{meta['data_shape'][0]} rows, {meta['elapsed_seconds'] / 60:.1f} min on {meta['mode']}"
)
print(f"Final log-joints: {meta['final_log_joints']}")
print()
print(f"{'Model':45s}  {'MAE':>6s}  {'RMSE':>6s}  {'R^2':>6s}")
print("-" * 70)
print(f"{'Transformer (2024-25, published)':45s}  {11.90:>6.2f}  {'   -':>6s}  {'   -':>6s}")
print(f"{'CNN-LSTM, Li 2018 (published)':45s}  {12.61:>6.2f}  {'   -':>6s}  {'   -':>6s}")
print(f"{'LSTM, Zheng 2017 (published)':45s}  {13.52:>6.2f}  {'   -':>6s}  {'   -':>6s}")
print(
    f"{'RandomForest (same training rows)':45s}  {baselines['random_forest']['mae']:>6.2f}  "
    f"{baselines['random_forest']['rmse']:>6.2f}  {baselines['random_forest']['r2']:>6.3f}"
)
print(
    f"{'Ridge (same training rows)':45s}  {baselines['ridge']['mae']:>6.2f}  "
    f"{baselines['ridge']['rmse']:>6.2f}  {baselines['ridge']['r2']:>6.3f}"
)
print(
    f"{'jaxcross BMA (all chains)':45s}  {bma['bma']['mae']:>6.2f}  "
    f"{bma['bma']['rmse']:>6.2f}  {bma['bma']['r2']:>6.3f}"
)
print(
    f"{'jaxcross best-chain-only':45s}  {best['point']['mae']:>6.2f}  "
    f"{best['point']['rmse']:>6.2f}  {best['point']['r2']:>6.3f}"
)
print()
print("CI coverage (nominal vs empirical):")
for level in (90, 95, 99):
    bma_c = bma[f"ci_{level}"]
    best_c = best[f"ci_{level}"]
    print(
        f"  {level}% CI:  BMA={bma_c['coverage']:.1%} (w={bma_c['avg_width']:.1f})  "
        f"best={best_c['coverage']:.1%} (w={best_c['avg_width']:.1f})"
    )

## Expected runtime on Kaggle 2×T4

- Clone + install: ~30 s
- Fetch + preprocess: ~20 s
- **Inference (8 chains × 300 sweeps × 20 631 rows, pmap):** ~45-60 min
- Evaluation + baselines: ~5 min
- Total notebook time: ~60-75 min, well within Kaggle's free-tier 12 h GPU budget.

## After the run finishes

Results persist in `/kaggle/working/jaxcross/examples/c_mapss/results/` and download as part of the notebook artifacts:

- `inference/FD001/chain_{0..7}.jxc, best_chain.jxc, log_joint_traces.npy, inference_meta.json`
- `evaluation/FD001/metrics.json, best_chain_metrics.json, rul_predictions.csv (+ .arrow)`
- `baselines/FD001/baseline_metrics.json`

Pull the notebook output locally and `cat examples/c_mapss/results/evaluation/FD001/metrics.json` to see the final numbers to drop into `FD001_COMPARISON.md`.